# Actividad — Sesión 3: Del dato crudo al dataset listo
### Ingestión y preparación de datos · Inteligencia Artificial (FP13)
**Nombre:** _______________________  **Matrícula:** __________  **Fecha:** 

---
### Instrucciones
Trabajas con los **mismos 4 archivos** de la clase (`flota_norteamerica.csv`, `flota_europa.xlsx`,
`flota_asia.txt`, `flota_datalake.parquet`), que deben estar en la **misma carpeta** que este
notebook. Completa cada bloque marcado con `# TODO`. Responde también las **preguntas de análisis**
(celdas de texto). 

**Entrega (hoy mismo):** sube a *Canvas* el link de github a este notebook ejecutado (menú *Kernel → Restart &
Run All* sin errores) con nombre `S3_ApellidoNombre.ipynb`.

**Objetivo:** dejar el dataset **consolidado, limpio y enriquecido** para responder la pregunta:
*¿qué características del powertrain determinan el consumo (L/100 km) de un vehículo?*

## Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

## Parte 1 · Ingestión y consolidación  *(2 pts)*
Lee cada archivo con la función adecuada y únelos en un solo DataFrame `df`.
Recuerda: el TXT usa separador `~` y marca los faltantes con `?`.

In [ ]:
# Lectura de las 4 fuentes, cada una con la función adecuada a su formato
df_csv = pd.read_csv('flota_norteamerica.csv')
df_xls = pd.read_excel('flota_europa.xlsx', sheet_name='registros')
df_txt = pd.read_csv('flota_asia.txt', sep='~', na_values='?')   # separador ~ y '?' = faltante
df_pq  = pd.read_parquet('flota_datalake.parquet')

for nombre, parte in [('csv', df_csv), ('xlsx', df_xls), ('txt', df_txt), ('parquet', df_pq)]:
    print(f'{nombre:>8}: {parte.shape}')

# Consolidación vertical (apilar filas). ignore_index=True reinicia el índice 0..N-1
df = pd.concat([df_csv, df_xls, df_txt, df_pq], ignore_index=True)

# 'horsepower' puede quedar como texto si alguna fuente traía '?' u otros símbolos:
# lo forzamos a numérico y lo que no se pueda convertir se vuelve NaN (se imputa en la Parte 2).
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

print('Consolidado ->', df.shape)   # se espera (410, 9)

## Parte 2 · Limpieza básica  *(2 pts)*
1. Reporta cuántos **duplicados** hay y elimínalos.
2. Reporta cuántos **faltantes** hay en `horsepower` e **imputa** con la mediana.

In [ ]:
# Duplicados: filas idénticas en TODAS sus columnas (mismo auto cargado dos veces)
print('Duplicados:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Faltantes en horsepower
print('Faltantes horsepower:', df['horsepower'].isna().sum())

# Imputamos con la MEDIANA (no la media) porque es robusta a valores extremos:
# unos pocos motores muy potentes inflarían el promedio.
mediana_hp = df['horsepower'].median()
print('Mediana usada para imputar:', mediana_hp)
df['horsepower'] = df['horsepower'].fillna(mediana_hp)

print('Filas finales:', len(df), '| Faltantes HP:', df['horsepower'].isna().sum())

## Parte 3 · Mapeos y unidades métricas  *(1.5 pts)*
1. Crea la columna `region` mapeando `origin` (1=EUA, 2=Europa, 3=Japón).
2. Crea `peso_kg` (1 lb = 0.453592 kg).
3. Crea `consumo_L_100km` a partir de `mpg` (fórmula: `235.215 / mpg`).

In [ ]:
# 1) Región: traducimos el código numérico de 'origin' a una etiqueta legible
mapa_region = {1: 'EUA', 2: 'Europa', 3: 'Japon'}
df['region'] = df['origin'].map(mapa_region)

# 2) Peso: el dataset viene en libras -> Sistema Internacional
LB_A_KG = 0.453592
df['peso_kg'] = df['weight'] * LB_A_KG

# 3) Consumo: mpg (mientras MÁS alto, mejor) -> L/100 km (mientras MÁS bajo, mejor).
#    Es una relación inversa, por eso la constante 235.215 va dividida entre mpg.
df['consumo_L_100km'] = 235.215 / df['mpg']

print(df['region'].value_counts(dropna=False))
df[['name', 'region', 'peso_kg', 'consumo_L_100km']].head()

## Parte 4 · Extracción e ingeniería de características  *(2 pts)*
1. Extrae la **marca** (primer token de `name`). Corrige al menos: `chevy`/`chevroelt`→`chevrolet`,
   `toyouta`→`toyota`, `vw`/`vokswagen`→`volkswagen`.
2. Crea la característica `relacion_pot_peso` = `horsepower / peso_kg`.
3. **Diseña tú una característica nueva** con sentido automotriz (por ejemplo, peso por cilindro,
   una bandera `es_importado`, etc.) y explícala en la celda de texto.

In [ ]:
# 1) Marca normalizada: primer token del nombre + diccionario de correcciones de captura
correcciones = {
    'chevy': 'chevrolet',
    'chevroelt': 'chevrolet',
    'toyouta': 'toyota',
    'vw': 'volkswagen',
    'vokswagen': 'volkswagen',
    'volkswagon': 'volkswagen',
    'maxda': 'mazda',
    'mercedes-benz': 'mercedes',
}
df['marca'] = (df['name']
               .str.strip()
               .str.lower()
               .str.split().str[0]
               .replace(correcciones))

# 2) Relación potencia-peso: cuántos HP mueve cada kg del vehículo
df['relacion_pot_peso'] = df['horsepower'] / df['peso_kg']

# 3) MI CARACTERÍSTICA: 'peso_por_cilindro' = kg de vehículo que debe mover cada cilindro
df['mi_feature'] = df['peso_kg'] / df['cylinders']   # peso_por_cilindro

print('Marcas distintas:', df['marca'].nunique())
print(df['marca'].value_counts().head(10))
df[['name', 'marca', 'relacion_pot_peso', 'mi_feature']].head()

**✍️ Explica tu característica nueva** (2–3 líneas: qué mide y por qué podría ayudar a predecir el consumo):

Mi característica es **`peso_por_cilindro` = `peso_kg` / `cylinders`**: mide cuántos kilogramos de
vehículo tiene que mover cada cilindro del motor, es decir, qué tan "exigido" está el powertrain
respecto a la masa que arrastra.

Debería ayudar a predecir el consumo porque captura el **dimensionamiento del motor frente al
vehículo**, algo que ni el peso ni el número de cilindros explican por separado. Un valor alto
indica un motor chico cargando demasiado peso: trabaja a mayor carga y con la mariposa más
abierta, lo que eleva el consumo. Un valor bajo indica un motor sobrado para el vehículo, que
consume mucho en vacío aunque vaya relajado. Por eso esperamos que se relacione con
`consumo_L_100km` de forma no trivial y aporte información distinta a la de las variables crudas.

## Parte 5 · Análisis de correlación  *(2 pts)*
1. Calcula la matriz de correlación de las variables numéricas relevantes.
2. Grafica el **mapa de calor**.
3. Identifica un **par altamente correlacionado** (`|r| > 0.9`) y decide cuál conservar.

In [ ]:
num = ['mpg', 'cylinders', 'horsepower', 'peso_kg', 'consumo_L_100km',
       'relacion_pot_peso', 'mi_feature']
corr = df[num].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=.5, square=True)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

# Listado automático de pares redundantes (|r| > 0.9) para justificar la respuesta
mascara = np.triu(np.ones(corr.shape), k=1).astype(bool)
pares = (corr.where(mascara)
             .stack()
             .rename('r')
             .reset_index()
             .rename(columns={'level_0': 'var_1', 'level_1': 'var_2'}))
print('Pares con |r| > 0.9:')
print(pares[pares['r'].abs() > 0.9].sort_values('r', key=abs, ascending=False))

**✍️ Pregunta de análisis:** menciona **un par** de variables con `|r| > 0.9`, indica cuál
**eliminarías** y **justifica** por qué (2–3 líneas):

El par más evidente es **`mpg` y `consumo_L_100km`**, con una correlación negativa muy fuerte
(|r| ≈ 0.9 o más). No es casualidad: `consumo_L_100km` se calculó como `235.215 / mpg`, así que
son **la misma variable expresada en dos unidades** y no aportan información independiente.

**Eliminaría `mpg`** y conservaría `consumo_L_100km`, porque es la variable objetivo del
planteamiento, está en unidades métricas (las que usamos en la región) y su relación con el peso y
la potencia es prácticamente lineal, mientras que `mpg` guarda una relación hiperbólica que
complica los modelos lineales. Mantener ambas provocaría **multicolinealidad**: coeficientes
inestables y difíciles de interpretar, además de fuga de información si `mpg` quedara como
predictora del propio consumo.

*(Otro par típico es `cylinders`–`peso_kg` / `horsepower`–`peso_kg`, donde conservaría `peso_kg`
por ser una medida continua y directa de la masa a mover.)*

## Cierre  *(reporte, 0.5 pts)*
Muestra la forma final del dataset y sus columnas.

In [ ]:
print('Dataset final:', df.shape)
print('Columnas:', list(df.columns))
df.head()

---
### Rúbrica (10 pts)
| Parte | Criterio | Pts |
|---|---|---|
| 1 | Lee los 4 formatos correctamente y consolida (410 filas) | 2.0 |
| 2 | Elimina 12 duplicados e imputa faltantes con mediana | 2.0 |
| 3 | Mapeos de región y conversiones métricas correctas | 1.5 |
| 4 | Marca normalizada + feature propia con justificación | 2.0 |
| 5 | Heatmap correcto + par redundante bien justificado | 2.0 |
| Cierre | Notebook corre completo sin errores (*Run All*) | 0.5 |
| | **Total** | **10** |

<br>

> Se evalúa que el notebook **corra de principio a fin sin errores** y que las **respuestas de
> análisis** estén redactadas con criterio de ingeniería.